In [1]:
#undef __noinline__

# Dot Product — Parallel Reduction

The dot product of two vectors A and B of length N is:

$$\text{result} = \sum_{i=0}^{N-1} A_i \cdot B_i$$

On the CPU this is a simple loop. On the GPU it splits into two stages:

1. **Partial products** — each thread multiplies pairs of elements and accumulates a local sum
2. **Parallel reduction** — threads inside a block cooperate to collapse all local sums into one

The reduction step requires two CUDA features we haven't seen yet:

| Keyword | What it does |
|---|---|
| `__shared__` | Declares memory shared by all threads in the same block (fast, on-chip) |
| `__syncthreads()` | Barrier — every thread in the block waits here until all have arrived |

The diagram below shows how the reduction collapses 256 values into 1 using a tree of additions.

![Dot Product](images/dot_product_cuda.png)

In [2]:
#include <algorithm>

const int N = 33 * 1024;
const int THREADS = 256;
const int BLOCKS = std::min(32, (N + THREADS - 1) / THREADS);

### The kernel

Each thread strides through the whole array accumulating a running `temp` sum, then
writes that into shared memory. After a `__syncthreads()` barrier the block performs
a tree reduction: in each step, the active threads halve and the surviving ones add
their neighbour's value. The block's final sum lands in `partial[blockIdx.x]`.

In [3]:
__global__ void dot_product(float *a, float *b, float *partial) {
    __shared__ float tile[THREADS];   // one float per thread, on-chip

    int tid    = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    int lane   = threadIdx.x;

    // Phase 1: each thread accumulates its share of element-wise products
    float running = 0.0f;
    for (int i = tid; i < N; i += stride)
        running += a[i] * b[i];
    tile[lane] = running;

    __syncthreads();   // ensure all threads have written before we read

    // Phase 2: tree reduction inside this block
    for (int half = blockDim.x / 2; half > 0; half >>= 1) {
        if (lane < half)
            tile[lane] += tile[lane + half];
        __syncthreads();
    }

    // Thread 0 holds the block's total — write it out
    if (lane == 0)
        partial[blockIdx.x] = tile[0];
}

Allocate input and output arrays on both host and device.

In [4]:
float *h_a, *h_b, *h_partial;
float *d_a, *d_b, *d_partial;

h_a       = (float*)malloc(N * sizeof(float));
h_b       = (float*)malloc(N * sizeof(float));
h_partial = (float*)malloc(BLOCKS * sizeof(float));

cudaMalloc((void**)&d_a,       N      * sizeof(float));
cudaMalloc((void**)&d_b,       N      * sizeof(float));
cudaMalloc((void**)&d_partial, BLOCKS * sizeof(float));

Fill the vectors (`a[i] = i`, `b[i] = 2i`), copy to GPU, launch, and retrieve the partial sums.

In [5]:
for (int i = 0; i < N; i++) {
    h_a[i] = (float)i;
    h_b[i] = (float)i * 2.0f;
}

cudaMemcpy(d_a, h_a, N * sizeof(float), cudaMemcpyHostToDevice);
cudaMemcpy(d_b, h_b, N * sizeof(float), cudaMemcpyHostToDevice);

dot_product<<<BLOCKS, THREADS>>>(d_a, d_b, d_partial);

cudaMemcpy(h_partial, d_partial, BLOCKS * sizeof(float), cudaMemcpyDeviceToHost);

Sum the `BLOCKS` partial results on the CPU to get the final answer.
Since `a[i] = i` and `b[i] = 2i`, the expected value is `2 × Σi²`.
The closed-form for that is `2 × N(N−1)(2N−1)/6` evaluated at N−1.

In [6]:
double gpu_result = 0.0;
for (int i = 0; i < BLOCKS; i++)
    gpu_result += h_partial[i];

// Closed-form reference: 2 * sum of i^2 for i = 0..N-1
double ref = 2.0 * (double)(N - 1) * N * (2.0 * (N - 1) + 1) / 6.0;

printf("GPU result : %.6g\n", gpu_result);
printf("Reference  : %.6g\n", ref);

GPU result : 2.57236e+13
Reference  : 2.57236e+13


In [7]:
cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_partial);

free(h_a);
free(h_b);
free(h_partial);